# Chapter 43: Kaggle Feature Engineering and Leakage Control (Advanced)

## Learning Objectives
- Build fold-safe target encoding
- Compare leaky vs non-leaky pipelines
- Measure fold variance
- Document leakage audit outcomes

## Prerequisites
- Chapters 0-37 completion recommended
- Python and ML fundamentals
- Optional: matplotlib for richer plots (ASCII fallback included)


# Chapter 43: Kaggle Feature Engineering and Leakage Control (Advanced)

## Why this chapter matters
Competition performance often depends more on leakage-safe feature engineering than on model complexity.

## Learning goals
1. Build fold-safe target encoding from scratch.
2. Create interaction and ratio features safely.
3. Detect leakage with split-aware diagnostics.
4. Visualize fold score variance.

## Advanced feature engineering checklist
1. domain transforms (log, clip, winsorize)
2. interaction terms (x1*x2, x1/x2)
3. group aggregates (mean/count by category)
4. fold-safe target encoding

## Leakage pitfalls
- target encoding on full dataset
- future leakage in temporal folds
- duplicate entity leakage across train/valid

## Assignment
1. Implement K-fold target encoding and compare with leaky encoding.
2. Plot fold score spread.
3. Build leakage audit table for each engineered feature.


In [ ]:
from __future__ import annotations

import csv
import math
import random
from collections import defaultdict
from typing import Dict, List, Tuple


def load_rows(path: str) -> List[Dict[str, str]]:
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


def kfold_indices(n: int, k: int = 5, seed: int = 42) -> List[List[int]]:
    idx = list(range(n))
    random.Random(seed).shuffle(idx)
    folds = [[] for _ in range(k)]
    for i, j in enumerate(idx):
        folds[i % k].append(j)
    return folds


def logistic_fit(X: List[List[float]], y: List[int], lr: float = 0.03, epochs: int = 250) -> List[float]:
    n, d = len(X), len(X[0])
    w = [0.0] * (d + 1)

    for _ in range(epochs):
        for i in range(n):
            z = w[0] + sum(w[j + 1] * X[i][j] for j in range(d))
            p = 1.0 / (1.0 + math.exp(-max(-35, min(35, z))))
            err = y[i] - p
            w[0] += lr * err
            for j in range(d):
                w[j + 1] += lr * err * X[i][j]
    return w


def logistic_predict_prob(X: List[List[float]], w: List[float]) -> List[float]:
    out = []
    d = len(X[0])
    for row in X:
        z = w[0] + sum(w[j + 1] * row[j] for j in range(d))
        p = 1.0 / (1.0 + math.exp(-max(-35, min(35, z))))
        out.append(p)
    return out


def auc_score(y_true: List[int], y_prob: List[float]) -> float:
    pos = [(p, y) for p, y in zip(y_prob, y_true) if y == 1]
    neg = [(p, y) for p, y in zip(y_prob, y_true) if y == 0]
    if not pos or not neg:
        return 0.5
    wins = 0.0
    total = len(pos) * len(neg)
    for pp, _ in pos:
        for pn, _ in neg:
            if pp > pn:
                wins += 1
            elif pp == pn:
                wins += 0.5
    return wins / total


def fold_safe_target_encoding(cat_vals: List[str], y: List[int], folds: List[List[int]]) -> List[float]:
    n = len(cat_vals)
    encoded = [0.0] * n
    global_mean = sum(y) / len(y)

    for valid_idx in folds:
        valid_set = set(valid_idx)
        sum_cnt = defaultdict(lambda: [0.0, 0])

        for i in range(n):
            if i in valid_set:
                continue
            c = cat_vals[i]
            sum_cnt[c][0] += y[i]
            sum_cnt[c][1] += 1

        for i in valid_idx:
            c = cat_vals[i]
            if sum_cnt[c][1] == 0:
                encoded[i] = global_mean
            else:
                encoded[i] = sum_cnt[c][0] / sum_cnt[c][1]

    return encoded


def leaky_target_encoding(cat_vals: List[str], y: List[int]) -> List[float]:
    sum_cnt = defaultdict(lambda: [0.0, 0])
    for c, t in zip(cat_vals, y):
        sum_cnt[c][0] += t
        sum_cnt[c][1] += 1
    return [sum_cnt[c][0] / sum_cnt[c][1] for c in cat_vals]


def maybe_plot_fold_scores(scores_safe: List[float], scores_leaky: List[float]) -> None:
    try:
        import matplotlib.pyplot as plt
    except Exception:
        print("Fold scores safe:", [round(s, 4) for s in scores_safe])
        print("Fold scores leaky:", [round(s, 4) for s in scores_leaky])
        return

    xs = list(range(1, len(scores_safe) + 1))
    plt.figure(figsize=(7, 4))
    plt.plot(xs, scores_safe, marker="o", label="fold-safe")
    plt.plot(xs, scores_leaky, marker="o", label="leaky")
    plt.xlabel("Fold")
    plt.ylabel("AUC")
    plt.title("Fold-safe vs Leaky Target Encoding")
    plt.legend()
    plt.tight_layout()
    plt.show()


def main() -> None:
    rows = load_rows("Y.Kaggle_Data/diabetes.csv")
    y = [int(float(r["Outcome"])) for r in rows]

    # Create a synthetic categorical feature from glucose ranges (for demo).
    cat_vals = []
    for r in rows:
        g = float(r["Glucose"])
        if g < 100:
            cat_vals.append("low")
        elif g < 140:
            cat_vals.append("mid")
        else:
            cat_vals.append("high")

    base_features = [[
        float(r["BMI"]),
        float(r["Age"]),
        float(r["Pregnancies"]),
    ] for r in rows]

    folds = kfold_indices(len(rows), k=5, seed=42)

    te_safe = fold_safe_target_encoding(cat_vals, y, folds)
    te_leaky = leaky_target_encoding(cat_vals, y)

    scores_safe, scores_leaky = [], []
    for valid in folds:
        valid_set = set(valid)
        train = [i for i in range(len(rows)) if i not in valid_set]

        X_train_safe = [base_features[i] + [te_safe[i]] for i in train]
        X_valid_safe = [base_features[i] + [te_safe[i]] for i in valid]

        X_train_leaky = [base_features[i] + [te_leaky[i]] for i in train]
        X_valid_leaky = [base_features[i] + [te_leaky[i]] for i in valid]

        y_train = [y[i] for i in train]
        y_valid = [y[i] for i in valid]

        w_safe = logistic_fit(X_train_safe, y_train)
        w_leaky = logistic_fit(X_train_leaky, y_train)

        p_safe = logistic_predict_prob(X_valid_safe, w_safe)
        p_leaky = logistic_predict_prob(X_valid_leaky, w_leaky)

        scores_safe.append(auc_score(y_valid, p_safe))
        scores_leaky.append(auc_score(y_valid, p_leaky))

    print("Mean AUC fold-safe:", round(sum(scores_safe) / len(scores_safe), 5))
    print("Mean AUC leaky    :", round(sum(scores_leaky) / len(scores_leaky), 5))

    maybe_plot_fold_scores(scores_safe, scores_leaky)


if __name__ == "__main__":
    main()


## Checkpoint

1. You can implement fold-safe encoders.
2. You can explain why full-data encoding is leaky.
3. You can compare fold score stability.

## Guided Exercise
Add smoothing to target encoding (`count`-based shrinkage) and compare AUC.

In [ ]:
# TODO (Student Exercise)
# Implement smoothed target encoding and benchmark against raw fold-safe encoding.

## Quick Quiz

**Q1.** Why is target encoding high-risk for leakage?

**Answer:** Target information can leak directly into features.

**Q2.** What is fold-safe encoding principle?

**Answer:** Each row’s encoding must be computed without that row’s fold targets.

**Q3.** Why monitor fold variance?

**Answer:** High variance signals instability/overfitting.


## Homework
Build a feature audit table with source, leakage risk, and mitigation strategy.